In [1]:
import os

def scan_files(root_dir=".", extensions=(".py",), exclude_dirs=None):
    if exclude_dirs is None:
        exclude_dirs = []

    # 🔥 Normalize excluded paths
    exclude_dirs = [os.path.normpath(os.path.join(root_dir, d)) for d in exclude_dirs]

    for dirpath, dirnames, filenames in os.walk(root_dir):
        # 🔥 Remove excluded directories based on FULL PATH
        dirnames[:] = [
            d for d in dirnames
            if os.path.normpath(os.path.join(dirpath, d)) not in exclude_dirs
        ]

        for filename in filenames:
            if filename.endswith(extensions):
                full_path = os.path.join(dirpath, filename)

                print("=" * 80)
                print(f"FILE: {full_path}")
                print("=" * 80)

                try:
                    with open(full_path, "r", encoding="utf-8") as f:
                        print(f.read())
                except Exception as e:
                    print(f"[ERROR] Could not read file: {e}")


if __name__ == "__main__":
    extensions_to_scan = (".py")
    root_directory = "."

    # ✅ Use forward slashes (cross-platform safe)
    excluded_folders = [
      
        
    ]

    scan_files(root_directory, extensions_to_scan, excluded_folders)

FILE: .\convert_hf_to_gguf.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from __future__ import annotations

import logging
import argparse
import contextlib
import json
import os
import re
import sys
from enum import IntEnum
from pathlib import Path
from hashlib import sha256
from typing import TYPE_CHECKING, Any, Callable, ContextManager, Iterable, Iterator, Literal, Sequence, TypeVar, cast
from itertools import chain

import math
import numpy as np
import torch

if TYPE_CHECKING:
    from torch import Tensor

if 'NO_LOCAL_GGUF' not in os.environ:
    sys.path.insert(1, str(Path(__file__).parent / 'gguf-py'))
import gguf

logger = logging.getLogger("hf-to-gguf")


###### MODEL DEFINITIONS ######

class SentencePieceTokenTypes(IntEnum):
    NORMAL = 1
    UNKNOWN = 2
    CONTROL = 3
    USER_DEFINED = 4
    UNUSED = 5
    BYTE = 6


AnyModel = TypeVar("AnyModel", bound="type[Model]")


class Model:
    _model_classes: dict[str, type[Model]] = {}

    dir_model: Path
    ftype: gg